# QUY TRÌNH 1 - THU THẬP DỮ LIỆU

## E-commerce Data Lakehouse trên Object Storage

Dataset:
Olist Brazilian E-Commerce Public Dataset

Mục tiêu:
- Thu thập dữ liệu bảng từ Olist
- Thu thập dữ liệu ảnh sản phẩm
- Lựa chọn các bảng cần thiết
- Kiểm tra dữ liệu ban đầu
- Kiểm tra dữ liệu ảnh ban đầu
- Chuẩn hóa tên file CSV
- Lưu dữ liệu bảng vào Raw
- Chuẩn bị dữ liệu cho bước Data Ingestion

In [21]:
import pandas as pd
from pathlib import Path

In [22]:
print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


In [23]:
# Xác định đường dẫn dự án
BASE_DIR = Path.cwd().resolve()

# Nếu notebook chạy trong thư mục notebooks
# thì lấy thư mục cha làm project root
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Các thư mục dữ liệu
SOURCE_DIR = BASE_DIR / "data" / "source"
IMAGE_DIR = BASE_DIR / "data" / "images" / "products"
RAW_DIR = BASE_DIR / "data" / "raw"

# Kiểm tra thư mục dữ liệu nguồn
if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục source: {SOURCE_DIR}"
    )

if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục ảnh: {IMAGE_DIR}"
    )

# Raw có thể chưa tồn tại nên tạo nếu cần
RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("BASE_DIR  :", BASE_DIR)
print("SOURCE_DIR:", SOURCE_DIR)
print("IMAGE_DIR :", IMAGE_DIR)
print("RAW_DIR   :", RAW_DIR)

BASE_DIR  : /Users/camlee05/development/E-commerce-Data-Lakehouse
SOURCE_DIR: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/source
IMAGE_DIR : /Users/camlee05/development/E-commerce-Data-Lakehouse/data/images/products
RAW_DIR   : /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw


In [24]:
# Khai báo các file CSV nguồn

DATASETS = {
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "payments": "olist_order_payments_dataset.csv"
}

print(f"Số dataset được khai báo: {len(DATASETS)}")

for name, file_name in DATASETS.items():
    print(f"- {name}: {file_name}")

Số dataset được khai báo: 6
- customers: olist_customers_dataset.csv
- products: olist_products_dataset.csv
- orders: olist_orders_dataset.csv
- order_items: olist_order_items_dataset.csv
- reviews: olist_order_reviews_dataset.csv
- payments: olist_order_payments_dataset.csv


In [25]:
# Kiểm tra các file CSV nguồn

missing_files = [
    file_name
    for file_name in DATASETS.values()
    if not (SOURCE_DIR / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Thiếu các file CSV nguồn: {missing_files}"
    )

print("✓ Đầy đủ các file CSV nguồn")

✓ Đầy đủ các file CSV nguồn


In [26]:
# Đọc các dataset CSV

datasets = {
    name: pd.read_csv(SOURCE_DIR / file_name)
    for name, file_name in DATASETS.items()
}

print(f"✓ Đã load {len(datasets)} datasets")

for name, df in datasets.items():
    print(
        f"- {name}: "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )

✓ Đã load 6 datasets
- customers: 99,441 rows × 5 columns
- products: 32,951 rows × 9 columns
- orders: 99,441 rows × 8 columns
- order_items: 112,650 rows × 7 columns
- reviews: 99,224 rows × 7 columns
- payments: 103,886 rows × 5 columns


In [27]:
# Các định dạng ảnh được hỗ trợ

SUPPORTED_IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}

# Thu thập danh sách ảnh sản phẩm
image_files = [
    file
    for file in IMAGE_DIR.iterdir()
    if file.is_file()
    and file.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
]

print(f"✓ Tổng số ảnh sản phẩm: {len(image_files):,}")

✓ Tổng số ảnh sản phẩm: 5


In [28]:
# Thu thập metadata ảnh sản phẩm

image_metadata = []

for image_path in image_files:
    stat = image_path.stat()

    image_metadata.append({
        "file_name": image_path.name,
        "extension": image_path.suffix.lower(),
        "size_bytes": stat.st_size,
        "size_kb": round(
            stat.st_size / 1024,
            2
        )
    })

image_metadata_df = pd.DataFrame(image_metadata)

image_metadata_df.head()

,file_name,extension,size_bytes,size_kb
0,product_005.jpg,.jpg,126543,123.58
1,product_004.jpg,.jpg,36381,35.53
2,product_001.jpg,.jpg,181972,177.71
3,product_003.jpg,.jpg,41352,40.38
4,product_002.jpg,.jpg,229085,223.72


In [29]:
# Tổng quan dữ liệu ảnh

print("========== IMAGE OVERVIEW ==========")

print(
    "Tổng số ảnh:",
    len(image_metadata_df)
)

if not image_metadata_df.empty:

    total_image_size = (
        image_metadata_df["size_bytes"].sum()
    )

    print(
        "Tổng dung lượng:",
        round(
            total_image_size / (1024 * 1024),
            2
        ),
        "MB"
    )

    print("\nĐịnh dạng ảnh:")

    display(
        image_metadata_df["extension"]
        .value_counts()
        .rename_axis("extension")
        .reset_index(name="count")
    )

========== IMAGE OVERVIEW ==========
Tổng số ảnh: 5
Tổng dung lượng: 0.59 MB

Định dạng ảnh:


,extension,count
0,.jpg,5


In [30]:
# Kiểm tra file ảnh rỗng

empty_images = image_metadata_df[
    image_metadata_df["size_bytes"] == 0
]

print(
    f"Số file ảnh rỗng: {len(empty_images)}"
)

if empty_images.empty:
    print("✓ Không phát hiện file ảnh rỗng")
else:
    display(empty_images)

Số file ảnh rỗng: 0
✓ Không phát hiện file ảnh rỗng


In [31]:
# Tổng quan dữ liệu các dataset
overview = []

for name, df in datasets.items():
    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

overview_df = pd.DataFrame(overview)

overview_df

,dataset,rows,columns
0,customers,99441,5
1,products,32951,9
2,orders,99441,8
3,order_items,112650,7
4,reviews,99224,7
5,payments,103886,5


In [32]:
# Kiểm tra schema và kiểu dữ liệu

schema_info = []

for name, df in datasets.items():

    for column in df.columns:

        schema_info.append({
            "dataset": name,
            "column": column,
            "dtype": str(df[column].dtype)
        })

schema_df = pd.DataFrame(schema_info)

schema_df

,dataset,column,dtype
0,customers,customer_id,str
1,customers,customer_unique_id,str
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,str
4,customers,customer_state,str
5,products,product_id,str
6,products,product_category_name,str
7,products,product_name_lenght,float64
8,products,product_description_lenght,float64
9,products,product_photos_qty,float64


In [33]:
# Hiển thị thông tin chi tiết về từng dataset
for name, df in datasets.items():
    print("=" * 60)
    print(f"DATASET: {name}")
    print("Số dòng:", len(df))
    print("Số cột:", len(df.columns))
    print("Các cột:")
    print(df.columns.tolist())

DATASET: customers
Số dòng: 99441
Số cột: 5
Các cột:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
DATASET: products
Số dòng: 32951
Số cột: 9
Các cột:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
DATASET: orders
Số dòng: 99441
Số cột: 8
Các cột:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
DATASET: order_items
Số dòng: 112650
Số cột: 7
Các cột:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
DATASET: reviews
Số dòng: 99224
Số cột: 7
Các cột:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [34]:
# Kiểm tra giá trị thiếu trong từng dataset
def missing_values(df):
    result = df.isnull().sum().reset_index()
    result.columns = ["Column", "Missing Values"]
    result["Missing %"] = (
        result["Missing Values"] / len(df) * 100
    ).round(2)
    
    return result[result["Missing Values"] > 0]

for name, df in datasets.items():
    print("=" * 60)
    print(f"Missing values - {name}")
    display(missing_values(df))

Missing values - customers


,Column,Missing Values,Missing %


Missing values - products


,Column,Missing Values,Missing %
1,product_category_name,610,1.85
2,product_name_lenght,610,1.85
3,product_description_lenght,610,1.85
4,product_photos_qty,610,1.85
5,product_weight_g,2,0.01
6,product_length_cm,2,0.01
7,product_height_cm,2,0.01
8,product_width_cm,2,0.01


Missing values - orders


,Column,Missing Values,Missing %
4,order_approved_at,160,0.16
5,order_delivered_carrier_date,1783,1.79
6,order_delivered_customer_date,2965,2.98


... (2 outputs left out)

Missing values - reviews


,Column,Missing Values,Missing %
3,review_comment_title,87656,88.34
4,review_comment_message,58247,58.70


Missing values - payments


,Column,Missing Values,Missing %


In [35]:
# Kiểm tra duplicate toàn dòng

duplicate_results = []

for name, df in datasets.items():

    duplicate_count = int(
        df.duplicated().sum()
    )

    duplicate_results.append({
        "dataset": name,
        "duplicate_rows": duplicate_count
    })

duplicate_df = pd.DataFrame(
    duplicate_results
)

duplicate_df

,dataset,duplicate_rows
0,customers,0
1,products,0
2,orders,0
3,order_items,0
4,reviews,0
5,payments,0


In [36]:
# Kiểm tra Primary Key

key_columns = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "reviews": "review_id",
}

pk_results = []

for name, column in key_columns.items():

    df = datasets[name]

    null_count = int(
        df[column].isna().sum()
    )

    duplicate_count = int(
        df[column].duplicated().sum()
    )

    pk_results.append({
        "dataset": name,
        "primary_key": column,
        "null_count": null_count,
        "duplicate_count": duplicate_count,
        "valid": (
            null_count == 0
            and duplicate_count == 0
        )
    })

pk_results_df = pd.DataFrame(
    pk_results
)

pk_results_df

,dataset,primary_key,null_count,duplicate_count,valid
0,customers,customer_id,0,0,True
1,products,product_id,0,0,True
2,orders,order_id,0,0,True
3,reviews,review_id,0,814,False


In [37]:
# Kiểm tra quan hệ giữa các bảng (foreign key relationships)
def check_foreign_key(
    child_df,
    parent_df,
    key_col
):
    parent_ids = set(
        parent_df[key_col].dropna()
    )

    invalid_mask = (
        child_df[key_col].notna()
        & ~child_df[key_col].isin(parent_ids)
    )

    invalid_rows = int(
        invalid_mask.sum()
    )

    invalid_keys = int(
        child_df.loc[
            invalid_mask,
            key_col
        ].nunique()
    )

    return {
        "invalid_rows": invalid_rows,
        "invalid_unique_keys": invalid_keys
    }

In [38]:
# Lưu các dataset CSV vào Raw Layer

for name, df in datasets.items():

    output_path = (
        RAW_DIR / f"{name}.csv"
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"✓ Đã lưu: {output_path}"
    )

✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/customers.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/products.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/orders.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/order_items.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/reviews.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/payments.csv


In [39]:
# Kiểm tra các CSV trong Raw Layer

raw_csv_files = sorted(
    RAW_DIR.glob("*.csv")
)

print(
    f"Số CSV trong Raw: "
    f"{len(raw_csv_files)}"
)

for file in raw_csv_files:
    print("-", file.name)

Số CSV trong Raw: 6
- customers.csv
- order_items.csv
- orders.csv
- payments.csv
- products.csv
- reviews.csv


In [40]:
# Tổng kết Data Collection

print(
    "========== DATA COLLECTION SUMMARY =========="
)

print(
    f"CSV datasets : {len(datasets)}"
)

print(
    f"Raw CSV files: {len(raw_csv_files)}"
)

print(
    f"Product images: {len(image_files)}"
)

print(
    f"Empty images: {len(empty_images)}"
)

collection_status = (
    "READY"
    if len(raw_csv_files) == len(DATASETS)
    and len(image_files) > 0
    else "NOT_READY"
)

print(
    f"Collection status: {collection_status}"
)

========== DATA COLLECTION SUMMARY ==========
CSV datasets : 6
Raw CSV files: 6
Product images: 5
Empty images: 0
Collection status: READY
